# 8.1 参数高效微调（PEFT）

在端侧设备上对模型进行微调和个性化适配，使模型更好地服务本地用户，同时保护用户数据隐私。参数高效微调（PEFT）仅训练极少量参数，大幅降低端侧训练的显存需求。PEFT是端侧个性化部署的核心技术。

## 什么是PEFT？

参数高效微调（Parameter-Efficient Fine-Tuning）冻结预训练模型的大部分参数，仅训练少量额外参数来实现任务适配。核心方法包括：
- **LoRA**：在权重矩阵旁添加低秩分解$\Delta W = BA$
- **Adapter**：在Transformer层中插入小型瓶颈网络
- **Prefix Tuning**：在注意力前添加可学习的虚拟token
- **QLoRA**：基座模型量化+LoRA微调
- **DoRA**：权重分解的LoRA变体，分解幅度和方向

## 为什么用PEFT？

1. **显存大幅降低**：仅训练<1%的参数，7B模型QLoRA微调仅需约3.5GB显存
2. **训练速度快**：参数少、梯度少，训练速度比全量微调快2-5倍
3. **多任务切换**：同一基座模型+不同LoRA适配器，按需切换
4. **数据隐私**：端侧微调数据不出设备
5. **存储高效**：每个LoRA适配器仅几MB~几十MB，远小于完整模型

## PEFT的数学原理

**LoRA**：权重增量分解为低秩矩阵
$$h = xW^T + x \cdot \frac{\alpha}{r} \cdot A^T B^T = x(W + \frac{\alpha}{r}BA)^T$$
其中$A \in \mathbb{R}^{r \times d_{in}}$，$B \in \mathbb{R}^{d_{out} \times r}$，$r \ll \min(d_{in}, d_{out})$
参数量：$r(d_{in} + d_{out}) \ll d_{in} \times d_{out}$

**QLoRA**：基座模型NF4量化+LoRA微调
$$h = x \cdot \text{dequant}(W_{\text{NF4}}) + x \cdot \frac{\alpha}{r} BA$$
NF4量化：$q_i = \sigma \cdot \Phi^{-1}(i/16)$，利用正态分布的分位数函数确定量化级别

**DoRA**：将权重分解为幅度和方向，LoRA仅调整方向
$$h = x \cdot (m \cdot \frac{W + \frac{\alpha}{r}BA}{\|W + \frac{\alpha}{r}BA\|_c})$$
其中$m$为可学习的幅度向量，$\|\cdot\|_c$为列归一化

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### LoRA / QLoRA

#### 什么是LoRA？

LoRA（Low-Rank Adaptation）冻结预训练权重$W$，添加低秩增量$\Delta W = BA$，仅训练$A$和$B$两个小矩阵。推理时可将$\Delta W$合并回$W$，零额外开销。

#### 为什么LoRA有效？

1. **参数极少**：$r(d_{in}+d_{out}) \ll d_{in} \times d_{out}$，通常仅训练<1%参数
2. **零推理开销**：合并后$W' = W + \frac{\alpha}{r}BA$，与原始线性层等价
3. **多任务切换**：同一基座模型+不同LoRA适配器，按需切换

#### LoRA的数学原理

$$h = xW^T + x \cdot \frac{\alpha}{r} \cdot A^T B^T = x\left(W + \frac{\alpha}{r}BA\right)^T$$

其中：
- $W \in \mathbb{R}^{d_{out} \times d_{in}}$：冻结的预训练权重
- $A \in \mathbb{R}^{r \times d_{in}}$：下投影矩阵，Kaiming初始化
- $B \in \mathbb{R}^{d_{out} \times r}$：上投影矩阵，零初始化
- $r$：LoRA秩（通常4-64）
- $\alpha$：缩放因子（通常16或32）
- $B$零初始化确保训练开始时$\Delta W = 0$

#### LoRA Rank选择策略

| Rank | 参数占比 | 适用场景 | 典型任务 |
|------|---------|---------|----------|
| 4 | ~0.05% | 简单任务、极度内存受限 | 风格迁移、简单QA |
| 8 | ~0.1% | 通用微调 | 指令跟随、摘要 |
| 16 | ~0.2% | 复杂任务 | 代码生成、推理 |
| 32-64 | ~0.4-0.8% | 高精度需求 | 数学推理、多语言 |

经验法则：从r=8开始，如果验证集loss不收敛则增大rank。
target_modules通常选择q_proj和v_proj（注意力层），复杂任务可扩展到k_proj、o_proj和FFN层。

In [ ]:
class LoRALinear(nn.Module):
    """LoRA线性层"""
    def __init__(self, in_features, out_features, rank=8, alpha=16):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01, requires_grad=False)
        self.bias = nn.Parameter(torch.zeros(out_features), requires_grad=False)
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        self.scaling = alpha / rank
        self.merged = False

    def merge(self):
        if not self.merged:
            self.weight.data += self.scaling * (self.lora_B @ self.lora_A)
            self.merged = True

    def forward(self, x):
        if self.merged:
            return F.linear(x, self.weight, self.bias)
        base = F.linear(x, self.weight, self.bias)
        lora = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base + lora

class LoRAModel(nn.Module):
    """LoRA微调模型"""
    def __init__(self, dim=64, n_classes=10, rank=8, alpha=16):
        super().__init__()
        self.fc1 = LoRALinear(dim, dim*2, rank, alpha)
        self.fc2 = LoRALinear(dim*2, dim, rank, alpha)
        self.fc3 = LoRALinear(dim, n_classes, rank, alpha)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

    def lora_params(self):
        return [p for n, p in self.named_parameters() if 'lora_' in n]

    def freeze_base(self):
        for n, p in self.named_parameters():
            if 'lora_' not in n:
                p.requires_grad = False

dim, n_classes = 64, 10
model = LoRAModel(dim, n_classes, rank=8, alpha=16)
model.freeze_base()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.lora_params())

x_data = torch.randn(256, dim)
y_data = torch.randint(0, n_classes, (256,))
dataset = torch.utils.data.TensorDataset(x_data, y_data)
loader = torch.utils.data.DataLoader(dataset, batch_size=32)

optimizer = torch.optim.Adam(model.lora_params(), lr=1e-3)
losses = []
for epoch in range(20):
    for x, y in loader:
        loss = F.cross_entropy(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

with torch.no_grad():
    acc_before_merge = (model(x_data).argmax(1) == y_data).float().mean()

model.fc1.merge()
model.fc2.merge()
model.fc3.merge()

with torch.no_grad():
    acc_after_merge = (model(x_data).argmax(1) == y_data).float().mean()

print(f"=== LoRA微调 ===")
print(f"总参数: {total_params:,}")
print(f"可训练参数: {trainable_params:,} ({trainable_params/total_params:.2%})")
print(f"冻结参数: {total_params - trainable_params:,}")
print(f"\n合并前准确率: {acc_before_merge:.4f}")
print(f"合并后准确率: {acc_after_merge:.4f}")
print(f"训练损失: {losses[0]:.4f} -> {losses[-1]:.4f}")

print(f"\n--- 不同rank的参数量 ---")
for r in [1, 2, 4, 8, 16, 32, 64]:
    m = LoRAModel(dim, n_classes, rank=r)
    tp = sum(p.numel() for p in m.parameters())
    lp = sum(p.numel() for p in m.lora_params())
    print(f"  rank={r:<3} LoRA参数={lp:<8} 占比={lp/tp:.2%}")

### Adapter & Prefix Tuning

#### 什么是Adapter？

在Transformer层中插入小型瓶颈网络（下投影-非线性-上投影），仅训练Adapter参数，冻结其余权重。

#### 什么是Prefix Tuning？

在注意力前添加可学习的虚拟token（prefix），通过prefix引导模型行为，不修改模型权重。

#### 为什么用Adapter/Prefix Tuning？

1. **Adapter**：模块化设计，每个任务一个Adapter，可即插即用
2. **Prefix Tuning**：完全不动权重，仅添加虚拟token，更轻量
3. **端侧友好**：额外参数少，适合端侧微调

#### 数学原理

**Adapter**：$h' = h + \text{FFN}_{\text{adapter}}(h)$，其中$\text{FFN}_{\text{adapter}}(h) = W_{up}\sigma(W_{down}h)$
- $W_{down} \in \mathbb{R}^{r \times d}$：下投影
- $W_{up} \in \mathbb{R}^{d \times r}$：上投影
- 残差连接确保初始时Adapter输出接近0

**Prefix Tuning**：在K和V前拼接可学习prefix：
$$K' = [P_k; K], \quad V' = [P_v; V]$$
其中$P_k, P_v \in \mathbb{R}^{l_p \times d}$为$l_p$个可学习prefix的Key和Value

#### DoRA：权重分解的LoRA变体

DoRA（Weight-Decomposed Low-Rank Adaptation）将权重分解为幅度$m$和方向$V$：
$$W = m \cdot V, \quad m \in \mathbb{R}^{d_{out}}, \quad V \in \mathbb{R}^{d_{out} \times d_{in}}$$
LoRA仅调整方向，幅度单独学习：
$$W' = m' \cdot \frac{V + \frac{\alpha}{r}BA}{\|V + \frac{\alpha}{r}BA\|_c}$$
DoRA在相同rank下比LoRA精度更高，因为幅度和方向的更新解耦，更接近全量微调的更新模式。

In [ ]:
class AdapterLayer(nn.Module):
    """Adapter层: 插入在Transformer层中的小型适配器"""
    def __init__(self, dim, bottleneck=16):
        super().__init__()
        self.down = nn.Linear(dim, bottleneck)
        self.up = nn.Linear(bottleneck, dim)
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return x + self.gate * self.up(F.relu(self.down(x)))

class PrefixTuning(nn.Module):
    """Prefix Tuning: 在输入前添加可训练的虚拟token"""
    def __init__(self, dim, prefix_length=10):
        super().__init__()
        self.prefix = nn.Parameter(torch.randn(prefix_length, dim) * 0.02)

    def forward(self, x):
        prefix = self.prefix.unsqueeze(0).expand(x.shape[0], -1, -1)
        return torch.cat([prefix, x], dim=1)

class AdapterTransformer(nn.Module):
    """带Adapter的Transformer"""
    def __init__(self, dim=64, n_layers=4, n_heads=4, adapter_bottleneck=16):
        super().__init__()
        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(nn.ModuleDict({
                'attn': nn.MultiheadAttention(dim, n_heads, batch_first=True),
                'ffn': nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim)),
                'adapter': AdapterLayer(dim, adapter_bottleneck),
                'ln1': nn.LayerNorm(dim),
                'ln2': nn.LayerNorm(dim),
            }))

    def forward(self, x):
        for layer in self.layers:
            h = layer['ln1'](x)
            h, _ = layer['attn'](h, h, h)
            x = x + h
            x = x + layer['adapter'](layer['ffn'](layer['ln2'](x)))
        return x

dim = 64
adapter_model = AdapterTransformer(dim, n_layers=4, n_heads=4, adapter_bottleneck=16)
prefix_tuner = PrefixTuning(dim, prefix_length=10)

adapter_params = sum(p.numel() for n, p in adapter_model.named_parameters() if 'adapter' in n)
total_adapter_params = sum(p.numel() for p in adapter_model.parameters())
prefix_params = sum(p.numel() for p in prefix_tuner.parameters())

x = torch.randn(2, 16, dim)
x_with_prefix = prefix_tuner(x)
out = adapter_model(x_with_prefix)

print(f"=== Adapter vs Prefix Tuning ===")
print(f"\nAdapter:")
print(f"  Adapter参数: {adapter_params:,} ({adapter_params/total_adapter_params:.2%})")
print(f"  瓶颈维度: 16")
print(f"  结构: 下投影->ReLU->上投影")
print(f"  门控初始化为0，训练初期不影响原始输出")

print(f"\nPrefix Tuning:")
print(f"  Prefix参数: {prefix_params:,}")
print(f"  Prefix长度: 10")
print(f"  输入形状: {x.shape} -> {x_with_prefix.shape}")

print(f"\n=== PEFT方法对比 ===")
methods = [
    ('LoRA', '低秩矩阵', f'r*(m+n)', '合并后零开销', '通用首选'),
    ('DoRA', '幅度+方向分解', f'r*(m+n)+m', '合并后零开销', '高精度需求'),
    ('Adapter', '瓶颈MLP', f'2*d*bn', '额外前向计算', '模块化部署'),
    ('Prefix Tuning', '虚拟token', f'len*d', '增加序列长度', '生成任务'),
    ('QLoRA', '量化+LoRA', f'r*(m+n)', '合并后零开销', '端侧微调'),
]
print(f"\n{'方法':<15} {'原理':<15} {'参数量':<15} {'推理开销':<15} {'适用场景':<15}")
print("-" * 75)
for name, principle, params, overhead, scene in methods:
    print(f"{name:<15} {principle:<15} {params:<15} {overhead:<15} {scene:<15}")

### QLoRA：量化基座 + LoRA微调

#### 什么是QLoRA？

QLoRA将基座模型量化为4bit（NF4格式），仅LoRA适配器保持高精度，使7B模型在24GB显存上可训练。核心创新：NF4量化格式、双重量化（double quantization）和分页优化器。

#### 为什么QLoRA有效？

1. **显存极低**：7B模型NF4量化后仅约3.5GB，加上LoRA和优化器状态约需12-16GB
2. **精度接近全量微调**：NF4是信息论最优的4bit量化，精度损失极小
3. **训练稳定**：分页优化器避免优化器状态的内存溢出

#### QLoRA的数学原理

前向计算：
$$h = x \cdot \text{dequant}(W_{\text{NF4}}) + x \cdot \frac{\alpha}{r} BA$$

**NF4量化**：利用正态分布的分位数函数确定量化级别：
$$q_i = \sigma \cdot \Phi^{-1}(i/16), \quad i = 0, 1, \ldots, 15$$

其中$\Phi^{-1}$为标准正态分布的逆CDF，$\sigma$为权重标准差。NF4的16个量化级别不是均匀分布的，而是集中在概率密度高的区域，因此比均匀INT4精度更高。

**双重量化**：对量化常量（scale和zero-point）再次量化为FP4，进一步节省存储。每个block的量化常量从32bit降至4bit，额外节省约0.37bit/param。

**分页优化器**：当GPU显存不足时，将优化器状态自动offload到CPU内存，需要时再加载回GPU。使用NVIDIA统一内存管理实现。

In [ ]:
class NF4Quantizer:
    """NF4（Normal Float 4-bit）量化器
    NF4假设权重服从正态分布，将量化点均匀分布在正态分布的分位数上，
    使得每个量化区间包含大致相同的概率质量，比均匀量化精度更高。
    16个量化级别（4bit），对应正态分布[-1,1]区间的16个分位数。"""
    def __init__(self):
        self.quantiles = torch.tensor([
            -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
            -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
            0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
            0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0
        ])

    def quantize(self, tensor):
        scale = tensor.abs().max() / 1.0
        scale = torch.clamp(scale, min=1e-8)
        normalized = tensor / scale
        indices = torch.zeros_like(normalized, dtype=torch.long)
        for i in range(len(self.quantiles) - 1):
            mask = (normalized >= self.quantiles[i]) & (normalized < self.quantiles[i + 1])
            indices[mask] = i
        indices[normalized >= self.quantiles[-1]] = len(self.quantiles) - 1
        return indices.to(torch.uint8), scale

    def dequantize(self, indices, scale):
        return self.quantiles[indices.long()] * scale

class QLoRALinear(nn.Module):
    """QLoRA线性层：NF4量化权重 + LoRA适配器"""
    def __init__(self, in_features, out_features, rank=8, alpha=16):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.scaling = alpha / rank

        self.nf4 = NF4Quantizer()
        weight = torch.randn(out_features, in_features) * 0.02
        self.register_buffer('weight_indices', torch.zeros(out_features, in_features, dtype=torch.uint8))
        self.register_buffer('weight_scale', torch.tensor(1.0))
        self.weight_indices, self.weight_scale = self.nf4.quantize(weight)

        self.lora_A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))
        self.merged = False

    def forward(self, x):
        weight_deq = self.nf4.dequantize(self.weight_indices, self.weight_scale)
        result = F.linear(x, weight_deq)
        if not self.merged:
            result += (x @ self.lora_A @ self.lora_B) * self.scaling
        return result

    def merge(self):
        weight_deq = self.nf4.dequantize(self.weight_indices, self.weight_scale)
        merged_weight = weight_deq + (self.lora_A @ self.lora_B).T * self.scaling
        self.weight_indices, self.weight_scale = self.nf4.quantize(merged_weight)
        self.merged = True

qlora_layer = QLoRALinear(128, 64, rank=4)
x = torch.randn(2, 128)

with torch.no_grad():
    out_before = qlora_layer(x)
    qlora_layer.merge()
    out_after = qlora_layer(x)
    diff = (out_before - out_after).abs().max()

base_weight_mem = 128 * 64 * 4
nf4_mem = 128 * 64 * 0.5
lora_mem = (128 * 4 + 4 * 64) * 4

print(f"=== QLoRA: 量化基座 + LoRA微调 ===")
print(f"基座权重: {128}×{64} = {128*64:,} 参数")
print(f"LoRA参数: rank={4}, A={128}×{4}, B={4}×{64} = {128*4+4*64:,} 参数")
print(f"LoRA占比: {(128*4+4*64)/(128*64)*100:.2f}%")
print(f"\n内存对比:")
print(f"  FP32基座: {base_weight_mem/1024:.1f} KB")
print(f"  NF4基座: {nf4_mem/1024:.1f} KB (节省{(1-nf4_mem/base_weight_mem)*100:.0f}%)")
print(f"  LoRA适配器(FP32): {lora_mem/1024:.1f} KB")
print(f"  QLoRA总计: {(nf4_mem+lora_mem)/1024:.1f} KB")
print(f"  vs FP32全量微调: {(nf4_mem+lora_mem)/base_weight_mem*100:.1f}%")
print(f"\n合并前后输出差异: {diff:.6f} (NF4再量化误差)")
print(f"注意: merge后重新NF4量化会引入额外误差，产业级部署通常保持分离或合并为FP16")

print(f"\nQLoRA核心优势:")
print(f"  1. NF4量化: 正态分布最优4bit格式，16个非均匀量化级别")
print(f"  2. 双重量化: 量化scale本身也量化为FP4，进一步节省内存")
print(f"  3. 分页优化器: 优化器状态在CPU/GPU间分页，降低峰值显存")

### 产业级实战：使用 HuggingFace PEFT 库

#### 什么是PEFT库？

HuggingFace生态的标准PEFT实现，支持LoRA、AdaLoRA、IA³、Prefix Tuning等方法，与transformers无缝集成。

#### 为什么用PEFT库？

1. **标准化接口**：统一的配置和训练接口，无需手写PEFT逻辑
2. **与transformers集成**：一行代码即可将PEFT应用到任何HuggingFace模型
3. **QLoRA支持**：配合bitsandbytes实现4bit量化+LoRA微调

#### 使用流程

1. 配置量化参数（BitsAndBytesConfig）
2. 加载量化模型（from_pretrained + quantization_config）
3. 配置LoRA参数（LoraConfig：r, lora_alpha, target_modules）
4. 包装模型（get_peft_model）
5. 训练（标准Trainer）
6. 合并权重（merge_and_unload）

#### LoraConfig关键参数

- `r`：LoRA秩，控制适配器容量（通常4-64）
- `lora_alpha`：缩放因子，控制适配器影响强度（通常16或32）
- `target_modules`：应用LoRA的模块名（如q_proj, v_proj）
- `lora_dropout`：Dropout率，防止过拟合

#### 多LoRA服务

产业级部署中，同一基座模型可加载多个LoRA适配器，按需切换：
- 每个适配器仅几MB~几十MB
- 切换时只需替换LoRA权重，无需重新加载基座
- 典型框架：vLLM、S-LoRA、LoRAX支持多LoRA并发服务

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import gc

MODEL_NAME = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32,
)
base_model.eval()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['c_attn'],
    bias='none',
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

total_params = sum(p.numel() for p in peft_model.parameters())
trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print(f"\n总参数: {total_params:,}")
print(f"可训练参数: {trainable_params:,}")
print(f"可训练比例: {trainable_params/total_params*100:.2f}%")

print(f"\nLoRA配置:")
print(f"  rank (r): {lora_config.r}")
print(f"  alpha: {lora_config.lora_alpha}")
print(f"  scaling: {lora_config.lora_alpha / lora_config.r}")
print(f"  target_modules: {lora_config.target_modules}")
print(f"  dropout: {lora_config.lora_dropout}")

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

peft_model.train()
optimizer = torch.optim.AdamW(peft_model.parameters(), lr=1e-4)

train_texts = [
    'The future of AI is bright and promising.',
    'Machine learning models are getting smaller and faster.',
    'On-device AI enables privacy-preserving inference.',
    'Quantization reduces model size with minimal accuracy loss.',
    'Edge computing brings AI closer to the user.',
]

encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=32, return_tensors='pt')
dataset = TensorDataset(encodings['input_ids'], encodings['attention_mask'])
loader = DataLoader(dataset, batch_size=2, shuffle=True)

peft_model.print_trainable_parameters()
print(f"\n开始LoRA微调...")
for epoch in range(3):
    total_loss = 0
    for batch_ids, batch_mask in loader:
        outputs = peft_model(input_ids=batch_ids, attention_mask=batch_mask, labels=batch_ids)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    print(f"  Epoch {epoch+1}/3, Loss: {total_loss/len(loader):.4f}")

peft_model.eval()
test_input = tokenizer('The future of AI is', return_tensors='pt')
with torch.no_grad():
    output = peft_model.generate(**test_input, max_new_tokens=15, do_sample=False)
print(f"\n微调后输出: {tokenizer.decode(output[0], skip_special_tokens=True)}")

In [ ]:
print(f"=== LoRA权重合并 (Merge) ===")
merged_model = peft_model.merge_and_unload()

test_input = tokenizer('The future of AI is', return_tensors='pt')
with torch.no_grad():
    output_merged = merged_model.generate(**test_input, max_new_tokens=15, do_sample=False)

text_merged = tokenizer.decode(output_merged[0], skip_special_tokens=True)
print(f"合并后输出: {text_merged}")

merged_params = sum(p.numel() for p in merged_model.parameters())
merged_mem = sum(p.numel() * p.element_size() for p in merged_model.parameters()) / 1024 / 1024
print(f"\n合并后参数量: {merged_params:,} (与原始模型相同)")
print(f"合并后内存: {merged_mem:.1f} MB")
print(f"\n产业界LoRA工作流:")
print(f"1. 训练: base_model + LoRA adapters (仅训练adapter参数)")
print(f"2. 保存: adapter_model.bin (仅几MB) + 可选保存合并后完整模型")
print(f"3. 部署: merge_and_unload() → 无额外推理开销")
print(f"4. 多任务: 同一base_model + 不同adapter → 按需切换")
print(f"5. 多LoRA服务: vLLM/S-LoRA支持并发服务多个LoRA适配器")

del merged_model, peft_model, base_model
gc.collect()

In [ ]:
print(f"=== 产业级QLoRA: bitsandbytes NF4 + PEFT LoRA ===")

nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=nf4_config,
    device_map='auto' if torch.cuda.is_available() else None,
)

qlora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=['c_attn'],
    bias='none',
)

qlora_model = get_peft_model(base_model_4bit, qlora_config)
qlora_model.print_trainable_parameters()

base_mem = sum(p.numel() * p.element_size() for p in base_model_4bit.parameters()) / 1024 / 1024
trainable = sum(p.numel() for p in qlora_model.parameters() if p.requires_grad)
print(f"\n基座模型(NF4)内存: {base_mem:.1f} MB")
print(f"可训练参数: {trainable:,}")
print(f"\nQLoRA产业级工作流:")
print(f"1. 加载: AutoModel.from_pretrained + BitsAndBytesConfig(load_in_4bit=True)")
print(f"2. 配置: LoraConfig + get_peft_model → 仅LoRA参数可训练")
print(f"3. 训练: SFTTrainer / Trainer → 标准训练循环")
print(f"4. 保存: model.save_pretrained() → 仅保存adapter权重")
print(f"5. 部署: merge_and_unload() → 重新量化或直接部署")
print(f"6. 多LoRA: vLLM/S-LoRA → 同一基座并发服务多个适配器")

del qlora_model, base_model_4bit
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

---
### LoRA组合与多LoRA服务

产业级部署中，同一基座模型需要服务多个不同任务，每个任务对应一个LoRA适配器。多LoRA服务（Multi-LoRA Serving）是关键基础设施。

#### LoRA组合策略

| 策略 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| **切换式** | 请求时加载对应LoRA | 简单 | 切换延迟 |
| **合并式** | 多LoRA加权合并到基座 | 无切换延迟 | 任务间干扰 |
| **并行式** | 基座+多LoRA同时驻留 | 零切换延迟 | 内存占用大 |
| **S-LoRA** | 分页管理LoRA权重 | 内存高效 | 实现复杂 |

#### LoRA合并的数学原理

多个LoRA适配器可以加权合并：

$$W_{\text{merged}} = W_0 + \sum_{i=1}^{K} \alpha_i \cdot B_i A_i$$

其中$\alpha_i$为第$i$个LoRA的合并权重。当$\alpha_i$为0-1变量时为选择式合并，当$\alpha_i$为连续值时为插值式合并。

#### S-LoRA：产业级多LoRA服务

S-LoRA的核心思想是将LoRA权重存储在GPU的统一内存中，按需分页加载到GPU HBM：

- **PagedAttention思想**：将LoRA权重分页管理，按需加载
- **统一内存利用**：GPU统一内存(>1TB)远大于HBM(80GB)，可存储数千个LoRA
- **Batch推理**：不同请求使用不同LoRA，在同一个batch中并行推理

| 方案 | 支持LoRA数 | 切换延迟 | 内存效率 |
|------|-----------|---------|----------|
| **vLLM Multi-LoRA** | ~100 | <1ms | 中 |
| **S-LoRA** | ~1000 | ~1ms | 高 |
| **LoRAX** | ~10000 | ~2ms | 极高 |

#### IA³：比LoRA更轻量的PEFT

IA³（Infused Adapter by Inhibiting and Amplifying Inner Activations）通过逐元素缩放激活值实现参数高效微调：

$$h' = l_k \odot h_k \quad \text{(Key缩放)}$$
$$v' = l_v \odot v \quad \text{(Value缩放)}$$
$$f' = l_f \odot f \quad \text{(FFN缩放)}$$

IA³的参数量仅为LoRA的1/10：
- LoRA(r=16): $2 \times 16 \times d = 32d$ 参数/层
- IA³: $3 \times d$ 参数/层（Key/Value/FFN各1个缩放向量）

| 方法 | 可训练参数 | 精度 | 适用场景 |
|------|-----------|------|----------|
| **LoRA** | 0.1-1% | 高 | 通用微调 |
| **IA³** | 0.01-0.1% | 中高 | 极端内存受限 |
| **AdaLoRA** | 0.1-1% | 高 | 自适应rank分配 |
| **DoRA** | 0.1-1% | 极高 | 分解幅度+方向 |

In [ ]:
class MultiLoraScheduler:
    """多LoRA服务调度器"""
    def __init__(self, base_model_size_mb, lora_size_mb, hbm_size_mb=80000):
        self.base_size = base_model_size_mb
        self.lora_size = lora_size_mb
        self.hbm_size = hbm_size_mb

    def estimate_capacity(self, strategy='s_lora'):
        """估算可服务的LoRA数量"""
        available = self.hbm_size - self.base_size
        if strategy == 'all_in_hbm':
            return available // self.lora_size
        elif strategy == 's_lora':
            hot_loras = available // self.lora_size
            total_in_um = 1000000 // self.lora_size
            return hot_loras, total_in_um
        return 0

    def estimate_switch_latency(self, strategy, lora_size_mb):
        """估算LoRA切换延迟"""
        latencies = {
            'all_in_hbm': 0.01,
            's_lora': lora_size_mb / 1000,
            'on_demand_load': lora_size_mb / 100,
        }
        return latencies.get(strategy, 1.0)

sched = MultiLoraScheduler(base_model_size_mb=3500, lora_size_mb=20, hbm_size_mb=80000)
print("=== 多LoRA服务分析 (7B INT4基座 + LoRA r=16) ===")

all_in = sched.estimate_capacity('all_in_hbm')
hot, total = sched.estimate_capacity('s_lora')
print(f"\n--- 容量估算 (HBM=80GB) ---")
print(f"  基座模型: {sched.base_size}MB")
print(f"  单个LoRA: {sched.lora_size}MB")
print(f"  全部驻留HBM: {all_in}个LoRA")
print(f"  S-LoRA(热驻留+统一内存): {hot}个热LoRA + {total}个总LoRA")

print(f"\n--- 切换延迟 ---")
for strategy in ['all_in_hbm', 's_lora', 'on_demand_load']:
    lat = sched.estimate_switch_latency(strategy, sched.lora_size)
    print(f"  {strategy}: {lat:.2f}ms")

print(f"\n--- IA³ vs LoRA参数量对比 ---")
hidden_dim = 4096
lora_r = 16
lora_params = 2 * lora_r * hidden_dim
ia3_params = 3 * hidden_dim
print(f"  LoRA(r={lora_r}): {lora_params:,} 参数/层 ({lora_params/hidden_dim:.0f}x hidden_dim)")
print(f"  IA³: {ia3_params:,} 参数/层 ({ia3_params/hidden_dim:.0f}x hidden_dim)")
print(f"  IA³参数量仅为LoRA的{ia3_params/lora_params:.1%}")

print(f"\n=== 产业实践要点 ===")
print(f"1. S-LoRA是产业级多LoRA服务的标准方案: 统一内存+分页管理")
print(f"2. LoRA合并(加权平均)适合静态场景: 无切换延迟但有任务干扰")
print(f"3. IA³适合极端内存受限: 参数量仅LoRA的1/10，精度略低")
print(f"4. DoRA(幅度+方向分解)是LoRA的升级版: 精度更高但稍复杂")
print(f"5. 端侧多LoRA: 通常<10个适配器，全部驻留HBM即可")
print(f"6. LoRA rank选择: r=8(简单任务), r=16(通用), r=32-64(复杂任务)")

---
## LoRA训练数据工程与超参数调优

产业级PEFT的效果80%取决于数据质量，20%取决于算法选择。数据工程和超参数调优是LoRA微调成功的关键。

### LoRA微调数据工程

| 阶段 | 关键操作 | 产业实践 |
|------|---------|----------|
| **数据收集** | 端侧用户交互日志 | 隐私合规前提下收集，数据不出设备 |
| **数据清洗** | 去重、过滤低质量样本 | MinHash去重、长度过滤、质量分类器 |
| **数据格式** | Alpaca/ShareGPT格式 | 统一instruction-input-output格式 |
| **数据配比** | 多任务数据混合比例 | 按任务重要性加权，避免主导任务压制 |
| **数据量** | 最少有效样本数 | 简单任务500-1000条，复杂任务5000+ |

### LoRA超参数调优指南

| 超参数 | 推荐值 | 调优方向 | 影响程度 |
|--------|-------|---------|----------|
| **学习率** | 1e-4 ~ 3e-4 | LoRA通常比全量微调高5-10x | ★★★★★ |
| **LoRA rank** | 8/16/32 | 复杂任务增大rank | ★★★★☆ |
| **LoRA alpha** | 2×rank | alpha/rank决定缩放因子 | ★★★☆☆ |
| **目标模块** | q_proj,v_proj 或 all-linear | 更多模块=更多参数=更好效果 | ★★★★☆ |
| **训练epoch** | 3-5 | 过拟合风险：监控验证集loss | ★★★☆☆ |
| **Warmup** | 10% steps | 防止初期梯度爆炸 | ★★☆☆☆ |
| **LR Scheduler** | Cosine | 比Constant更稳定 | ★★☆☆☆ |
| **Batch Size** | 8-32 | 梯度累积模拟大batch | ★★★☆☆ |

### LoRA的局限性与替代方案

| 局限 | 原因 | 替代方案 |
|------|------|----------|
| **新知识注入困难** | 低秩约束限制表达能力 | 增大rank或使用全量微调 |
| **多任务干扰** | 不同任务的LoRA方向冲突 | 多LoRA独立服务或DoRA |
| **长上下文适应差** | LoRA主要修改注意力模式 | 增加embedding层LoRA |
| **量化后精度下降** | NF4量化引入额外误差 | 使用FP16基座或GPTQ/AWQ量化 |

---
## 总结与最佳实践

### PEFT技术选择决策树

```
端侧微调需求?
├── 内存充足(>8GB) → QLoRA (NF4+LoRA, 最佳性价比)
├── 内存紧张(4-8GB) → LoRA + INT8基座
├── 极端内存(<4GB) → IA³ 或 Prefix Tuning
├── 需要最高精度 → DoRA (幅度+方向分解)
└── 多任务服务 → S-LoRA (统一内存+分页管理)
```

### PEFT产业级Checklist

- [ ] 选择PEFT方法（LoRA/QLoRA/DoRA/IA³）
- [ ] 确定LoRA rank和目标模块
- [ ] 准备高质量训练数据（清洗+去重+格式化）
- [ ] 设置超参数（学习率、epoch、scheduler）
- [ ] 训练并监控验证集loss，防止过拟合
- [ ] 合并LoRA权重或保持分离部署
- [ ] 精度验证：微调后模型在目标任务和通用任务上的表现
- [ ] 多LoRA场景：实现适配器管理和热切换
- [ ] A/B测试：对比微调前后的用户体验指标
- [ ] 版本管理：LoRA适配器的版本追踪和回滚机制